# PRÁCTICA 2 PLN
# Desarrollo de una aplicación de Procesamiento del Lenguaje Natural

Alumnos:

Javier García Fernández

Miguel Ángel Véliz Ayala

# 6. Detección de contenido inapropiado usando ZSL, FSL y Chain-of-thought

In [ ]:
!pip install praw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 8.2 MB/s eta 0:00:00


In [ ]:
import json
import os
import random
import time
import logging
from datetime import datetime, timedelta
import praw
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Configuración del logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('reddit_detector')

# PARTE 1: EXTRACCIÓN DE DATOS
# Esta función extrae comentarios de Reddit de un subreddit específico.
def extract_reddit_comments(
    subreddit_name="OpinionesPolemicas",
    num_threads=10,          # Según el enunciado: 10 hilos
    comments_per_thread=50,  # Según el enunciado: 50 comentarios
    output_file=None,
    client_id="CLIENT_ID",          # ESTOS 3 PARAMETROS DEBEN SER REMPLAZADOS 
    client_secret="CLIENT_SECRET",  # POR LOS VALORES OBTENIDOS AL CREAR LA 
    user_agent="USER_AGENT"         # APLICACIÓN EN REDDIT

):

    if output_file is None:
        output_file = f"comments_{subreddit_name}.json"

    logger.info(f"Iniciando extracción de comentarios de r/{subreddit_name}")

    try:
        # Configuración para Reddit
        reddit = praw.Reddit(
            client_id=client_id,
            client_secret=client_secret,
            user_agent=user_agent
        )

        # Verificar si se puede acceder a la API
        try:
            test_subreddit = reddit.subreddit("test")
            next(test_subreddit.hot(limit=1), None)
            logger.info("Conexión a Reddit API exitosa")
        except Exception as e:
            logger.error(f"Error de autenticación en Reddit API: {e}")
            logger.error("Verifica tus credenciales de Reddit API")
            return None

        # Verificar si el subreddit existe
        try:
            target_subreddit = reddit.subreddit(subreddit_name)
            subreddit_info = target_subreddit.display_name
            logger.info(f"Subreddit encontrado: r/{subreddit_info}")
        except Exception as e:
            logger.error(f"Error al acceder al subreddit r/{subreddit_name}: {e}")
            return None

        # Definir el período de tiempo (último año)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=365)
        start_timestamp = int(start_date.timestamp())
        end_timestamp = int(end_date.timestamp())

        # Parámetros según el enunciado
        LIMIT_HILOS = num_threads
        COMENTARIOS_HILO = comments_per_thread

        # Valores mínimos para validar un comentario (reducidos para aumentar posibilidades)
        min_comment_len = 10
        n_words = 5

        # Inicializamos nuestro conjunto de datos
        data = []
        total_comments = 0
        total_threads = 0

        # Intentamos extraer de hot, new y top para aumentar probabilidades
        for sort_method in ["hot", "new", "top"]:
            if total_threads >= LIMIT_HILOS:
                break

            logger.info(f"Buscando hilos en {sort_method}...")

            if sort_method == "hot":
                submissions = target_subreddit.hot(limit=50)
            elif sort_method == "new":
                submissions = target_subreddit.new(limit=50)
            else:
                submissions = target_subreddit.top("all", limit=50)

            for submission in submissions:
                if total_threads >= LIMIT_HILOS:
                    break

                # Filtramos los hilos por fecha
                if submission.created_utc < start_timestamp or submission.created_utc > end_timestamp:
                    continue

                logger.info(f"Procesando hilo: {submission.title[:30]}...")

                try:
                    submission.comments.replace_more(limit=0)
                    comments = submission.comments.list()
                    comments_thread = len(comments)

                    logger.info(f"  - Encontrados {comments_thread} comentarios en este hilo")

                    if comments_thread == 0:
                        continue

                except Exception as e:
                    logger.warning(f"Error al expandir comentarios: {e}")
                    continue

                # Filtrar comentarios válidos
                valid_comments = []
                for comment in comments:
                    try:
                        if (len(comment.body) >= min_comment_len and
                            len(comment.body.split()) >= n_words):
                            valid_comments.append(comment)
                    except Exception:
                        continue

                logger.info(f"  - {len(valid_comments)} comentarios válidos encontrados")

                # Si no hay suficientes comentarios, pasamos al siguiente hilo
                if len(valid_comments) < min(5, COMENTARIOS_HILO):  # Al menos 5 comentarios
                    logger.info("  - No hay suficientes comentarios válidos, saltando hilo")
                    continue

                # Seleccionar aleatoriamente comentarios
                selected_comments = random.sample(
                    valid_comments,
                    min(COMENTARIOS_HILO, len(valid_comments))
                )

                # Crear estructura de datos para este hilo
                thread_data = {
                    "flair": submission.link_flair_text,
                    "title": submission.title,
                    "author": submission.author.name if submission.author else "Anonymous",
                    "date": submission.created_utc,
                    "score": submission.score,
                    "description": submission.selftext,
                    "comments": []
                }

                # Añadir comentarios seleccionados
                for comment in selected_comments:
                    try:
                        thread_data["comments"].append({
                            "user": comment.author.name if comment.author else "Anonymous",
                            "comment": comment.body,
                            "score": comment.score,
                            "date": comment.created_utc
                        })
                    except Exception as e:
                        continue

                data.append(thread_data)
                total_comments += len(thread_data["comments"])
                total_threads += 1

                logger.info(f"  - Hilo #{total_threads} procesado con éxito ({len(thread_data['comments'])} comentarios)")

                # Esperar para evitar rate limiting
                time.sleep(0.5)

        # Resumen final
        logger.info(f"Extracción completada: {total_threads} hilos, {total_comments} comentarios")

        # Si no hay datos, intentamos con un subreddit alternativo
        if total_threads == 0:
            logger.warning("No se encontraron hilos que cumplan los criterios")
            return None

        # Guardar en JSON
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

        logger.info(f"Datos guardados en {output_file}")
        return data

    except Exception as e:
        logger.error(f"Error general en la extracción de datos: {e}")
        return None

In [ ]:
# PARTE 2: CARGA DEL MODELO DE LENGUAJE
# Esta función carga el modelo de lenguaje pequeño, en nuestro caso gemma
def load_model(model_name="google/gemma-2b"):
    logger.info(f"Cargando modelo {model_name}...")

    try:
        # Configuración de dispositivo
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Usando dispositivo: {device}")

        # Cargar el tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # En CPU, intentar cargar en int8 para optimizar memoria
        if device == "cpu":
            logger.info("Cargando en precisión reducida para CPU...")
            try:
                from transformers import BitsAndBytesConfig

                # Configuración para reducir memoria
                quantization_config = BitsAndBytesConfig(
                    load_in_8bit=True,
                    llm_int8_threshold=6.0
                )

                model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    device_map="auto",
                    torch_dtype="auto",
                    quantization_config=quantization_config
                )
            except ImportError:
                # Si bitsandbytes no está disponible, cargar con optimizaciones estándar
                model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    device_map="auto",
                    torch_dtype="auto",
                    low_cpu_mem_usage=True
                )
        else:
            # En GPU, cargar normalmente
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                torch_dtype="auto"
            )

        logger.info(f"Modelo {model_name} cargado correctamente!")
        return model, tokenizer, model_name

    except Exception as e:
        logger.error(f"No se pudo cargar {model_name}: {e}")
        raise ValueError(f"No se pudo cargar el modelo {model_name}. Verifica la instalación de transformers y/o prueba con GPU.")


In [ ]:
# PARTE 3: TÉCNICAS DE DETECCIÓN DE CONTENIDO INAPROPIADO
# Esta función intenta detectar contenido inapropiado sin ejemplos
def detect_inappropriate_content_zsl(model, tokenizer, comment_text):
    # Acortar comentarios muy largos
    if len(comment_text) > 500:
        comment_text = comment_text[:500] + "..."

    prompt = f"""
    Por favor, analiza el siguiente comentario y determina si contiene contenido inapropiado o sensible.
    Un comentario inapropiado puede contener lenguaje ofensivo, odio, discriminación, amenazas,
    contenido sexual explícito, incitación a la violencia o cualquier otro contenido perjudicial.

    Comentario: "{comment_text}"

    ¿Este comentario contiene contenido inapropiado? Responde con "SÍ" o "NO".
    """

    try:
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=512)

        # Mover a dispositivo adecuado
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                temperature=0.7
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Buscar la respuesta final
        if 'Responde con "SÍ" o "NO".' in response:
            response = response.split('Responde con "SÍ" o "NO".')[-1].strip().upper()
        elif "?" in response:
            response = response.split("?")[-1].strip().upper()
        else:
            response = response.upper()

        # Normalizar la respuesta
        if "SÍ" in response or "SI" in response or "YES" in response:
            return True
        elif "NO" in response:
            return False
        else:
            # Búsqueda de palabras clave como última opción
            lower_response = response.lower()
            for keyword in ["inapropiado", "ofensivo", "inadecuado", "sensible"]:
                if keyword in lower_response:
                    return True
            return False

    except Exception as e:
        logger.error(f"Error en ZSL: {e}")
        return False

# Esta otra función hace lo mismo pero detectando con ejemplos de ambas categorías
def detect_inappropriate_content_fsl(model, tokenizer, comment_text):
    # Acortar comentarios muy largos
    if len(comment_text) > 500:
        comment_text = comment_text[:500] + "..."

    # Ejemplos para el aprendizaje con pocos ejemplos
    prompt = f"""
    A continuación se presentan ejemplos de comentarios y su clasificación como apropiados o inapropiados:

    Comentario: "Me parece que este político es incompetente y sus políticas son desastrosas."
    Clasificación: Apropiado (Es una opinión crítica pero no contiene lenguaje ofensivo o discriminatorio)

    Comentario: "Todos los [grupo étnico] son unos delincuentes y deberían ser expulsados del país."
    Clasificación: Inapropiado (Contiene discriminación étnica y generalizaciones ofensivas)

    Comentario: "No estoy de acuerdo con esta ley porque creo que limita nuestras libertades."
    Clasificación: Apropiado (Expresa desacuerdo político sin lenguaje ofensivo)

    Comentario: "Ojalá le pase algo malo a ese político que odio, se lo merece por [insulto]."
    Clasificación: Inapropiado (Contiene deseos de daño y lenguaje ofensivo)

    Comentario: "Este tema es una mierda. No entiendo cómo pueden defenderlo."
    Clasificación: Inapropiado (Contiene lenguaje vulgar y ofensivo)

    Ahora, clasifica el siguiente comentario:
    Comentario: "{comment_text}"
    Clasificación:
    """

    try:
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=768)

        # Mover a dispositivo adecuado
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                temperature=0.7
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extraer la clasificación
        if "Clasificación:" in response:
            classification = response.split("Clasificación:")[-1].strip().lower()
        else:
            classification = response.lower()

        # Determinar si es inapropiado basado en palabras clave
        inappropriate_terms = ["inapropiado", "ofensivo", "inadecuado", "sensible", "objetable"]
        is_inappropriate = any(term in classification for term in inappropriate_terms)

        # Verificar negaciones que podrían invertir el significado
        if is_inappropriate:
            negations = ["no es", "no contiene", "no se considera"]
            for negation in negations:
                for term in inappropriate_terms:
                    if f"{negation} {term}" in classification:
                        is_inappropriate = False
                        break

        return is_inappropriate

    except Exception as e:
        logger.error(f"Error en FSL: {e}")
        return False

# Función que razona paso a paso para detectar contenido inapropiado
def detect_inappropriate_content_cot(model, tokenizer, comment_text):
    # Acortar comentarios muy largos
    if len(comment_text) > 500:
        comment_text = comment_text[:500] + "..."

    # Prompt para razonamiento paso a paso
    prompt = f"""
    Analiza paso a paso el siguiente comentario para determinar si contiene contenido inapropiado o sensible:

    Comentario: "{comment_text}"

    Sigue estos pasos para el análisis:
    1. Identifica si hay lenguaje ofensivo o insultos.
    2. Verifica si hay discriminación basada en raza, género, religión u otras características.
    3. Detecta si hay amenazas, incitación a la violencia o contenido que promueva el odio.
    4. Comprueba si hay contenido sexual explícito o inapropiado.

    Proporciona tu análisis para cada paso y luego tu conclusión final:
    ¿El comentario contiene contenido inapropiado? Responde con "SÍ" o "NO".
    """

    try:
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=768)

        # Mover a dispositivo adecuado
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,  # Más tokens para permitir razonamiento completo
                do_sample=False,
                temperature=0.5
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Buscar conclusión final
        conclusion_patterns = [
            "¿El comentario contiene contenido inapropiado?",
            "Conclusión:",
            "En conclusión,"
        ]

        conclusion = ""
        for pattern in conclusion_patterns:
            if pattern in response:
                parts = response.split(pattern)
                if len(parts) > 1:
                    conclusion = parts[-1].strip().upper()
                    break

        # Si hay conclusión clara
        if conclusion:
            if any(positive in conclusion for positive in ["SÍ", "SI", "YES", "INAPROPIADO"]):
                return True
            elif "NO" in conclusion or "APROPIADO" in conclusion:
                return False

        # Análisis del razonamiento si no hay conclusión clara
        lower_response = response.lower()

        # Contar indicadores de contenido inapropiado
        inappropriate_indicators = [
            "ofensivo", "insulto", "discrimina", "racista", "sexista",
            "amenaza", "violencia", "sexual", "explícito", "odio"
        ]

        # Patrones de negación
        negation_patterns = [
            "no es", "no hay", "no contiene", "no se detecta",
            "no se encuentra", "ausencia de"
        ]

        # Contar ocurrencias
        positive_count = 0
        negative_count = 0

        for indicator in inappropriate_indicators:
            positive_count += lower_response.count(indicator)

            for negation in negation_patterns:
                negative_pattern = f"{negation} {indicator}"
                negative_count += lower_response.count(negative_pattern)

        # Calcular score final
        final_score = positive_count - (negative_count * 2)

        return final_score > 0

    except Exception as e:
        logger.error(f"Error en CoT: {e}")
        return False

In [ ]:
# PARTE 4: ANÁLISIS DE COMENTARIOS
# Esta función analiza los comentarios de Reddit utilizando las tres técnicas
# de detección de la parte anterior
def analyze_reddit_data(data, model=None, tokenizer=None, model_name=None):
    if model is None or tokenizer is None:
        model, tokenizer, model_name = load_model()

    results = []
    total_comments = 0

    for thread_idx, thread in enumerate(data):
        logger.info(f"Analizando hilo {thread_idx+1}/{len(data)}: {thread['title'][:30]}...")
        thread_results = {
            "title": thread["title"],
            "comments": []
        }

        for comment_idx, comment in enumerate(thread["comments"]):
            logger.info(f"  - Procesando comentario {comment_idx+1}/{len(thread['comments'])}")
            comment_text = comment["comment"]
            total_comments += 1

            # Aplicar las tres técnicas de detección
            try:
                zsl_result = detect_inappropriate_content_zsl(model, tokenizer, comment_text)
                logger.info(f"    - ZSL: {'Inapropiado' if zsl_result else 'Apropiado'}")

                fsl_result = detect_inappropriate_content_fsl(model, tokenizer, comment_text)
                logger.info(f"    - FSL: {'Inapropiado' if fsl_result else 'Apropiado'}")

                cot_result = detect_inappropriate_content_cot(model, tokenizer, comment_text)
                logger.info(f"    - CoT: {'Inapropiado' if cot_result else 'Apropiado'}")

                comment_result = {
                    "comment": comment_text,
                    "zsl_result": zsl_result,
                    "fsl_result": fsl_result,
                    "cot_result": cot_result
                }
                thread_results["comments"].append(comment_result)

            except Exception as e:
                logger.error(f"Error al analizar comentario: {e}")

        results.append(thread_results)

    # Guardar resultados
    output_file = f"content_analysis_results_{model_name.split('/')[-1]}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    logger.info(f"Análisis completado ({total_comments} comentarios) y guardado en {output_file}")
    return results

In [ ]:
# PARTE 5: SELECCIÓN DE EJEMPLOS PARA ANÁLISIS FINAL
# Esta función selecciona ejemplos de comentarios clasificados como
# apropiados e inapropiados para un análisis posterior
def select_examples_for_analysis(results, num_examples=10):
    """
    Selecciona ejemplos de comentarios clasificados como apropiados e inapropiados para análisis
    """
    appropriate_comments = []
    inappropriate_comments = []

    for thread in results:
        for comment in thread["comments"]:
            # Calculamos un "score" de inapropiado basado en las tres técnicas
            inappropriate_score = sum([
                1 if comment["zsl_result"] else 0,
                1 if comment["fsl_result"] else 0,
                1 if comment["cot_result"] else 0
            ])

            comment_with_score = {
                "comment": comment["comment"],
                "zsl_result": comment["zsl_result"],
                "fsl_result": comment["fsl_result"],
                "cot_result": comment["cot_result"],
                "inappropriate_score": inappropriate_score
            }

            # Consideramos inapropiado si al menos 2 técnicas lo clasifican como tal
            if inappropriate_score >= 2:
                inappropriate_comments.append(comment_with_score)
            else:
                appropriate_comments.append(comment_with_score)

    # Seleccionamos aleatoriamente ejemplos de cada categoría
    selected_appropriate = random.sample(
        appropriate_comments,
        min(num_examples, len(appropriate_comments))
    )

    selected_inappropriate = random.sample(
        inappropriate_comments,
        min(num_examples, len(inappropriate_comments))
    )

    analysis_data = {
        "appropriate_comments": selected_appropriate,
        "inappropriate_comments": selected_inappropriate
    }

    # Guardar ejemplos seleccionados
    with open("selected_examples.json", "w", encoding="utf-8") as f:
        json.dump(analysis_data, f, indent=4, ensure_ascii=False)

    logger.info(f"Ejemplos seleccionados guardados en selected_examples.json")
    return analysis_data

In [ ]:
# PARTE 6: ANÁLISIS DE RENDIMIENTO DE LAS TÉCNICAS
# Esta función nos permitirá analizar el rendimiento de cada técnica en los ejemplos seleccionados
def analyze_technique_performance(selected_examples):
    # Inicializamos contadores
    techniques = ["zsl", "fsl", "cot"]
    stats = {
        "appropriate": {tech: {"correct": 0, "total": len(selected_examples["appropriate_comments"])} for tech in techniques},
        "inappropriate": {tech: {"correct": 0, "total": len(selected_examples["inappropriate_comments"])} for tech in techniques}
    }

    # Analizamos comentarios apropiados
    for comment in selected_examples["appropriate_comments"]:
        if not comment["zsl_result"]:
            stats["appropriate"]["zsl"]["correct"] += 1
        if not comment["fsl_result"]:
            stats["appropriate"]["fsl"]["correct"] += 1
        if not comment["cot_result"]:
            stats["appropriate"]["cot"]["correct"] += 1

    # Analizamos comentarios inapropiados
    for comment in selected_examples["inappropriate_comments"]:
        if comment["zsl_result"]:
            stats["inappropriate"]["zsl"]["correct"] += 1
        if comment["fsl_result"]:
            stats["inappropriate"]["fsl"]["correct"] += 1
        if comment["cot_result"]:
            stats["inappropriate"]["cot"]["correct"] += 1

    # Calculamos la precisión para cada técnica
    results = {}
    for tech in techniques:
        correct_appropriate = stats["appropriate"][tech]["correct"]
        correct_inappropriate = stats["inappropriate"][tech]["correct"]
        total_appropriate = stats["appropriate"][tech]["total"]
        total_inappropriate = stats["inappropriate"][tech]["total"]

        # Evitar división por cero
        if total_appropriate == 0:
            appropriate_accuracy = 0
        else:
            appropriate_accuracy = correct_appropriate / total_appropriate

        if total_inappropriate == 0:
            inappropriate_accuracy = 0
        else:
            inappropriate_accuracy = correct_inappropriate / total_inappropriate

        # Precisión total
        total_correct = correct_appropriate + correct_inappropriate
        total_examples = total_appropriate + total_inappropriate

        if total_examples == 0:
            accuracy = 0
        else:
            accuracy = total_correct / total_examples

        results[tech] = {
            "overall_accuracy": accuracy,
            "appropriate_accuracy": appropriate_accuracy,
            "inappropriate_accuracy": inappropriate_accuracy
        }

    # Guardar resultados de rendimiento
    with open("technique_performance.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    logger.info("Análisis de rendimiento guardado en technique_performance.json")

    # Mostrar resultados en consola
    logger.info("\n=== ANÁLISIS DE RESULTADOS ===")
    for tech in techniques:
        logger.info(f"Técnica: {tech.upper()}")
        logger.info(f"  - Precisión total: {results[tech]['overall_accuracy']:.2f}")
        logger.info(f"  - Precisión comentarios apropiados: {results[tech]['appropriate_accuracy']:.2f}")
        logger.info(f"  - Precisión comentarios inapropiados: {results[tech]['inappropriate_accuracy']:.2f}")

    return results

In [ ]:
def main():
    # 1. Verificar si ya existen datos extraídos
    data_file = "comments_OpinionesPolemicas.json"
    if os.path.exists(data_file):
        logger.info(f"Cargando datos existentes de {data_file}")
        try:
            with open(data_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Verificar si hay suficientes datos
            total_comments = sum(len(thread["comments"]) for thread in data)
            logger.info(f"Datos cargados: {len(data)} hilos, {total_comments} comentarios")

            # Si no hay datos suficientes, extraer datos específicamente de OpinionesPolemicas
            if len(data) < 10 or total_comments < 50:
                logger.warning("Datos insuficientes, intentando extraer más...")
                additional_data = extract_reddit_comments(subreddit_name="OpinionesPolemicas")
                if additional_data and len(additional_data) > 0:
                    # Combinar los datos existentes con los nuevos
                    data.extend(additional_data)
                    logger.info(f"Datos adicionales obtenidos de r/OpinionesPolemicas")
        except Exception as e:
            logger.error(f"Error al cargar {data_file}: {e}")
            data = extract_reddit_comments(subreddit_name="OpinionesPolemicas")
    else:
        # Extraer datos del subreddit OpinionesPolemicas
        data = extract_reddit_comments(subreddit_name="OpinionesPolemicas")

    # Si no hay datos suficientes después de intentar extraer, salir
    if not data or len(data) < 10:
        logger.error("No se pudieron obtener suficientes datos de r/OpinionesPolemicas. Revisa conexión o credenciales.")
        return

    # Asegurar que solo utilizamos 10 hilos
    data = data[:10]

    # Asegurar que cada hilo tenga máximo 50 comentarios
    for thread in data:
        if len(thread["comments"]) > 50:
            thread["comments"] = thread["comments"][:50]

    # 2. Cargar modelo (usando un SLM como especifica el enunciado)
    try:
        # Usar un modelo pequeño según el enunciado (Gema 2B o Llama 3.2 1B)
        model, tokenizer, model_name = load_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    except Exception as e:
        logger.error(f"Error al cargar el modelo: {e}")
        return

    # 3. Analizar comentarios
    try:
        results = []
        total_comments = 0
        batch_size = 10  # Tamaño de lote para comentarios

        for thread_idx, thread in enumerate(data):
            logger.info(f"Analizando hilo {thread_idx+1}/{len(data)}: {thread['title'][:30]}...")
            thread_results = {
                "title": thread["title"],
                "comments": []
            }

            # Procesar comentarios en lotes
            for i in range(0, len(thread["comments"]), batch_size):
                comment_batch = thread["comments"][i:i + batch_size]
                logger.info(f"  - Procesando lote de {len(comment_batch)} comentarios")

                for comment in comment_batch:
                    comment_text = comment["comment"]
                    total_comments += 1

                    # Aplicar las tres técnicas de detección
                    try:
                        zsl_result = detect_inappropriate_content_zsl(model, tokenizer, comment_text)
                        fsl_result = detect_inappropriate_content_fsl(model, tokenizer, comment_text)
                        cot_result = detect_inappropriate_content_cot(model, tokenizer, comment_text)

                        comment_result = {
                            "comment": comment_text,
                            "zsl_result": zsl_result,
                            "fsl_result": fsl_result,
                            "cot_result": cot_result
                        }
                        thread_results["comments"].append(comment_result)

                    except Exception as e:
                        logger.error(f"Error al analizar comentario: {e}")

                # Liberar memoria después de cada lote
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            results.append(thread_results)

        # Guardar resultados
        output_file = f"content_analysis_results_{model_name.split('/')[-1]}.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

        logger.info(f"Análisis completado ({total_comments} comentarios) y guardado en {output_file}")
    except Exception as e:
        logger.error(f"Error en el análisis de comentarios: {e}")
        return

    # 4. Seleccionar ejemplos para análisis final (10 como indica el enunciado)
    try:
        selected_examples = select_examples_for_analysis(results, num_examples=10)
    except Exception as e:
        logger.error(f"Error al seleccionar ejemplos: {e}")
        return

    # 5. Analizar rendimiento de las técnicas
    try:
        technique_performance = analyze_technique_performance(selected_examples)
    except Exception as e:
        logger.error(f"Error al analizar rendimiento: {e}")
        return

    # 6. Generar informe detallado
    try:
        # Implementamos directamente la generación del informe
        report = "# Informe de Detección de Contenido Inapropiado\n\n"

        # Resumen de rendimiento de técnicas
        report += "## Rendimiento de Técnicas\n\n"
        report += "| Técnica | Precisión Total | Precisión Apropiados | Precisión Inapropiados |\n"
        report += "|---------|----------------|---------------------|------------------------|\n"

        for tech, metrics in technique_performance.items():
            report += f"| {tech.upper()} | {metrics['overall_accuracy']:.2f} | {metrics['appropriate_accuracy']:.2f} | {metrics['inappropriate_accuracy']:.2f} |\n"

        # Ejemplos de comentarios apropiados
        report += "\n## Ejemplos de Comentarios Apropiados\n\n"
        for i, comment in enumerate(selected_examples["appropriate_comments"]):
            report += f"### Ejemplo {i+1}\n"
            report += f"```\n{comment['comment']}\n```\n"
            report += f"- ZSL: {'Inapropiado' if comment['zsl_result'] else 'Apropiado'}\n"
            report += f"- FSL: {'Inapropiado' if comment['fsl_result'] else 'Apropiado'}\n"
            report += f"- CoT: {'Inapropiado' if comment['cot_result'] else 'Apropiado'}\n\n"

        # Ejemplos de comentarios inapropiados
        report += "\n## Ejemplos de Comentarios Inapropiados\n\n"
        for i, comment in enumerate(selected_examples["inappropriate_comments"]):
            report += f"### Ejemplo {i+1}\n"
            report += f"```\n{comment['comment']}\n```\n"
            report += f"- ZSL: {'Inapropiado' if comment['zsl_result'] else 'Apropiado'}\n"
            report += f"- FSL: {'Inapropiado' if comment['fsl_result'] else 'Apropiado'}\n"
            report += f"- CoT: {'Inapropiado' if comment['cot_result'] else 'Apropiado'}\n\n"

        # Guardar informe
        with open("informe_deteccion_contenido.md", "w", encoding="utf-8") as f:
            f.write(report)

    # 7. Imprimir el informe por pantalla
        print("\n" + "="*80)
        print("INFORME DE DETECCIÓN DE CONTENIDO INAPROPIADO")
        print("="*80 + "\n")
        print(report)
        print("\n" + "="*80)

        logger.info("Informe de análisis generado en 'informe_deteccion_contenido.md'")
    except Exception as e:
        logger.error(f"Error al generar el informe: {e}")

if __name__ == "__main__":
    main()

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



INFORME DE DETECCIÓN DE CONTENIDO INAPROPIADO

# Informe de Detección de Contenido Inapropiado

## Rendimiento de Técnicas

| Técnica | Precisión Total | Precisión Apropiados | Precisión Inapropiados |
|---------|----------------|---------------------|------------------------|
| ZSL | 1.00 | 1.00 | 1.00 |
| FSL | 0.65 | 1.00 | 0.30 |
| COT | 0.50 | 0.00 | 1.00 |

## Ejemplos de Comentarios Apropiados

### Ejemplo 1
```
Pero el perro dura menos, el hijo es para siempre 🤣
```
- ZSL: Apropiado
- FSL: Apropiado
- CoT: Inapropiado

### Ejemplo 2
```
La parte que no entiendo es porque, si saben que el comunismo no existe, le echan la culpa al comunismo de todas las cosas malas de la existencia.
```
- ZSL: Apropiado
- FSL: Apropiado
- CoT: Inapropiado

### Ejemplo 3
```
No es una pérdida de tiempo. Es simplemente un entretenimiento.

Por supuesto que prefiero una relación con amor, pero eso tarda en llegar. Años, incluso decadas.
```
- ZSL: Apropiado
- FSL: Apropiado
- CoT: Inapropiado

### 

A continuación, vamos a realizar un análisis detallado de la apliación que hemos hecho de los modelos de lenguaje pequeños (Small Large Models, SLMs) para la detección automática de contenido inapropiado del subreddit `OpinionesPolemicas`. En concreto, vamos a evaluar tres estrategias distintas: Zero-Shot Learning (ZSL), Few-Shot Learning (FSL) y Chain-of-Thought (CoT), comparando su eficacia en esta tarea de clasificación binaria.

Para ello hemos recopilado 10 hilos con 50 comentarios cada uno, seleccionando posteriormente 20 comentarios representativos, 10 apropiados y 10 inapropiados para analizar el rendimiento de cada técnica. Si nos centramos un poco más en las técnicas mencionadas anteriormente, estas son:

- Zero-Shot Learning (ZSL): Se instruye al modelo para clasificar comentarios sin proporcionar ejemplos previos.
- Few-Shot Learning (FSL): Se proporcionan algunos ejemplos etiquetados para guiar la clasificación.
- Chain-of-Thought (CoT): Se solicita al modelo que razone paso a paso sobre sus decisiones.

Después de esta breve introducción, vamos a evaluar los resultados obtenidos.

---

Los resultados muestran diferencias significativas en el desempeño de cada técnica:

| Técnica | Precisión Total | Precisión Apropiados | Precisión Inapropiados |
|---------|------------------|-----------------------|--------------------------|
| ZSL     | 1.00             | 1.00                  | 1.00                     |
| FSL     | 0.65             | 1.00                  | 0.30                     |
| COT     | 0.50             | 0.00                  | 1.00                     |


Para entender mejor estos resultados, será mejor que calculemos algunas métricas adicionales como el F1-score para cada técnica:

## Fórmula general del F1-score

$$
\text{F1-score} = 2 \times \frac{\text{precision} \times \text{recall}}{\text{precision} + \text{recall}}
$$

### Para ZSL:

$$
\text{F1-score}_{\text{ZSL}} = 2 \times \frac{1.00 \times 1.00}{1.00 + 1.00} = 1.00
$$

### Para FSL:

$$
\text{F1-score}_{\text{FSL}} = 2 \times \frac{0.65 \times 0.65}{0.65 + 0.65} = 0.65
$$

### Para CoT:

$$
\text{F1-score}_{\text{CoT}} = 2 \times \frac{0.50 \times 0.50}{0.50 + 0.50} = 0.50
$$


Por último, también podemos expresar los resultados en forma de matriz de confusión, para representar visualmente el rendimiento de cada técnica:

### Matriz de confusión - ZSL

|                        | Predicción Apropiado | Predicción Inapropiado |
|------------------------|----------------------|--------------------------|
| **Real Apropiado**     | 10                    | 0                        |
| **Real Inapropiado**   | 0                    | 10                        |

### Matriz de confusión - FSL

|                        | Predicción Apropiado | Predicción Inapropiado |
|------------------------|----------------------|--------------------------|
| **Real Apropiado**     | 10                    | 0                        |
| **Real Inapropiado**   | 7                    | 3                        |

### Matriz de confusión - CoT

|                        | Predicción Apropiado | Predicción Inapropiado |
|------------------------|----------------------|--------------------------|
| **Real Apropiado**     | 0                    | 10                        |
| **Real Inapropiado**   | 0                    | 10                        |

---

Finalmente nos queda la parte más importante e interesante, analizar los resultados obtenidos. Lo haremos para cada técnica.

## Zero-Shot Learning (ZSL)

El enfoque ZSL muestra un rendimiento sorprendentemente perfecto (precisión de 1.00) en la clasificación de comentarios tanto apropiados como inapropiados. Esto sugiere que:

1. El modelo tiene una comprensión robusta de lo que constituye contenido inapropiado incluso sin ejemplos específicos.
2. Los comentarios seleccionados posiblemente presentan características muy distintivas que facilitan su clasificación.
3. Podría existir algún sesgo no identificado en la selección de la muestra que favoreció a esta técnica.

La perfección del ZSL es inusual en tareas de NLP, lo que sugiere que podría ser necesario validar estos resultados con conjuntos de datos más amplios y diversos.

## Few-Shot Learning (FSL)

El FSL muestra un comportamiento interesante:

1. Precisión perfecta (1.00) para contenidos apropiados: El modelo identifica correctamente todos los contenidos apropiados.
2. Baja precisión (0.30) para contenidos inapropiados: El modelo clasifica erróneamente el 70% de los contenidos inapropiados como apropiados.

Este sesgo hacia clasificar contenido como "apropiado" podría deberse a varios factores:

- La naturaleza y calidad de los ejemplos proporcionados durante el few-shot learning.
- Posible ambigüedad en algunos comentarios inapropiados que no presentan señales explícitas de contenido problemático.
- Una tendencia conservadora del modelo para evitar falsos positivos (etiquetar incorrectamente como inapropiado).

## Chain-of-Thought (CoT)

El enfoque CoT presenta un comportamiento prácticamente opuesto al FSL:

1. Precisión nula (0.00) para contenidos apropiados: Clasifica todos los contenidos apropiados como inapropiados.
2. Precisión perfecta (1.00) para contenidos inapropiados: Identifica correctamente todos los contenidos inapropiados.

Este sesgo extremo hacia clasificar todo como "inapropiado" podría explicarse por:

- El proceso de razonamiento paso a paso podría estar magnificando aspectos potencialmente problemáticos incluso en contenido benigno.
- La naturaleza reflexiva de CoT podría llevar al modelo a ser excesivamente cauteloso, prefiriendo falsos positivos sobre falsos negativos.
- Posibles sesgos en las instrucciones proporcionadas para el razonamiento.

---

Para completar el análisis de estas técnicas, podemos ver ejemplos concretos de cada tipo.

## Ejemplos "apropiados" mal clasificados por CoT

`Pero el perro dura menos, el hijo es para siempre 🤣`

El razonamiento del modelo probablemente interpretó la comparación entre mascotas e hijos como potencialmente ofensiva, cuando en realidad es una observación humorística sobre la longevidad de las relaciones

`La parte que no entiendo es porque, si saben que el comunismo no existe, le echan la culpa al comunismo de todas las cosas malas de la existencia.`

Este comentario político fue etiquetado incorrectamente como inapropiado. El razonamiento CoT posiblemente consideró el tema político como divisivo, aunque el comentario en sí no contiene lenguaje ofensivo o inapropiado.

## Ejemplos Inapropiados Mal Clasificados por FSL:

`Esa respuesta solo demuestra que o tienes problemas de aprendizaje o eres hipocrita y sinceramente estoy bien con cualquiera de las opciones.`

FSL clasificó incorrectamente este comentario como apropiado, a pesar de contener insultos personales ("problemas de aprendizaje", "hipócrita"). Esto sugiere que FSL podría no reconocer adecuadamente ataques personales sutiles.

`Ya salieron los gordos "esto no es verdadero comunismo" cuando sus gobiernos comunistas no salen como esperaban.`

Este comentario contiene lenguaje despectivo ("gordos") pero fue clasificado como apropiado por FSL, sugiriendo una dificultad para identificar insultos dirigidos a grupos

---

También podemos comparar de manera más matemática las técnicas. Podemos cuantificar el balance entre precisión y exhaustividad utilizando el índice Kappa de Cohen, que mide la concordancia entre clasificaciones teniendo en cuenta el azar:

## Fórmula del coeficiente Kappa de Cohen

$$
\kappa = \frac{1 - p_e}{1 - p_o}
$$

Donde:
- $p_o$ es la proporción de acuerdos observados.
- $p_e$ es la proporción de acuerdos esperados por azar.

También podemos calcular la tasa de error para cada técnica:

## Fórmula de la Tasa de Error

$$
\text{Error Rate} = 1 - \text{Accuracy}
$$

### Tasa de Error por Técnica

| Técnica | Error Rate |
|---------|------------|
| ZSL     | 0.00       |
| FSL     | 0.35       |
| CoT     | 0.50       |

---

Como conclusión de cada enfoque, podemos decir que:

- ZSL destaca como la técnica más efectiva para esta tarea específica, logrando una precisión perfecta que, aunque prometedora, debe validarse con conjuntos de datos más amplios, pues demasiado sospechosa.
- FSL muestra un sesgo hacia clasificar contenido como apropiado, lo que podría ser problemático en aplicaciones donde minimizar contenido inapropiado es prioritario.
- CoT muestra un sesgo hacia clasificar contenido como inapropiado, lo que resultaría en una experiencia restrictiva con muchos falsos positivos.


El bajo desempeño de FSL quizás se pueda deber a que los ejemplos son demasiado simplificados y en la realidad (o almenos en los comentarios encontrados) las personas usan señales más sutiles.

En cuanto al desempeño pobre de CoT, quizás el permitir 200 tokens nuevos no sea bueno, quzás esto sobreajusta, o mejor dicho, hace al modelo sobrepensar demasiado algo sencillo, encontrando o incluso generando problemas donde no los hay.